# Barley QTL Nergena — `barley_qtl_32593_32742`

|                        |                                               |
| ---------------------- | --------------------------------------------- |
| **Crop**               | Barley                                        |
| **Location**           | Nergena, Netherlands                          |
| **Year**               | 2025                                          |
| **Measurements**       | 1,680                                         |
| **Genotypes**          | 272 genotypes (Offspring, Elite line, Parent) |
| **Owner**              | Olivia Kacheyo                                |
| **Protocol**           | UNZA_PIRK_DIRK_LightPotential_14              |
| **PhotosynQ projects** | 32593, 32742                                  |

**Experiment:** QTL mapping population field trial for barley at Nergena (Wageningen campus). Genotype joined via Olivia's conversion key (plot line → genotype).

**Computed columns (25):** Same UNZA_PIRK_DIRK protocol — see Dataset 1.

**Additional column:** `sample_raw` (VARIANT).

In [ ]:
%pip install wadler-lindig -q

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.data import load_barley_qtl

df = load_barley_qtl(Path("data"))
print(f"Shape: {df.shape}")
df.columns.tolist()

In [ ]:
df.head()

## Exploratory Data Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4), layout="constrained")
df["genotype_type"].value_counts().plot(kind="bar", ax=ax)
ax.set(title="Measurements by genotype type", xlabel="Genotype type", ylabel="Count")
plt.show()

In [ ]:
kde_cols = [
    "phi2_ambient", "phi2_high", "LEF_ambient",
    "LEF_high", "LEF_light_potential", "PAR",
    "SPAD", "ambient_Temperature", "leaf_angle",
]

fig, axs = plt.subplots(3, 3, figsize=(12, 12), layout="constrained")
for ax, col in zip(axs.flatten(), kde_cols):
    df[col].dropna().plot(kind="kde", ax=ax)
    ax.set(title=col)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4), layout="constrained")
sns.boxplot(data=df, x="genotype_type", y="phi2_ambient", ax=ax)
ax.set(title="phi2_ambient by genotype type")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4), layout="constrained")
sns.boxplot(data=df, x="genotype_type", y="LEF_light_potential", ax=ax)
ax.set(title="LEF_light_potential by genotype type")
plt.show()

In [ ]:
df.describe()

## Correlation Heatmap

In [ ]:
phenotype_cols = [
    "phi2_ambient", "phi2_high", "LEF_ambient", "LEF_high",
    "LEF_light_potential", "PAR", "SPAD", "ambient_Temperature",
    "leaf_temperature_differential", "PIRK_amp_ambient", "PIRK_amp_high",
]

corr = df[phenotype_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8), layout="constrained")
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-1, vmax=1, ax=ax)
ax.set(title="Phenotype correlation matrix")
plt.show()

## Genotype Ranking

In [ ]:
genotype_means = df.groupby("genotype")[["phi2_ambient", "LEF_light_potential"]].mean()
top15 = genotype_means.nlargest(15, "phi2_ambient")

fig, axs = plt.subplots(1, 2, figsize=(12, 6), layout="constrained")

top15["phi2_ambient"].sort_values().plot(kind="barh", ax=axs[0])
axs[0].set(title="Top 15 genotypes — phi2_ambient", xlabel="Mean phi2_ambient")

top15["LEF_light_potential"].sort_values().plot(kind="barh", ax=axs[1])
axs[1].set(title="Top 15 genotypes — LEF_light_potential", xlabel="Mean LEF_light_potential")

plt.show()

## Heritability

In [ ]:
from src import drop_na_multiple, heritability, heritability_with_covariates

phenotypes = df[["phi2_ambient", "phi2_high", "LEF_ambient", "LEF_high", "LEF_light_potential"]]
gtype = df["genotype"]

data, gtype_clean = drop_na_multiple(phenotypes, gtype)

# Keep only genotypes with more than 3 entries
counts = gtype_clean.value_counts()
keep = counts[counts > 3].index
mask = gtype_clean.isin(keep)
data = data[mask]
gtype_clean = gtype_clean[mask]

h2 = heritability(data=data, gtype=gtype_clean)
h2.plot()
plt.show()
h2.as_frame()

In [ ]:
env_factors = df.loc[data.index, ["block"]]

h2_cov = heritability_with_covariates(data=data, env_factors=env_factors, gtype=gtype_clean)
h2_cov.plot()
plt.show()
h2_cov.as_frame()